# 05 · Transformer 心智模型：从字符串到下一个 token

> **学习目标**：把「string → tokens → ids → embeddings → ??? → logits → next token」**自己跑一遍**。能口头复述每一步在干嘛。
>
> **预备**：03 + 04 已过。
>
> **为什么重要**：所有 LLM 应用都是这条链子的变形。这条链子在你脑里清晰之后，看任何模型代码都不再陌生。

**整张图**：

```
  '我爱 NLP'
       │
       │  tokenizer
       ▼
  ['我', '爱', ' N', 'LP']         <- 子词，可能更碎
       │
       │  vocab 查表
       ▼
  [102, 423, 45, 1209]               <- token ids
       │
       │  nn.Embedding(vocab, d_model)
       ▼
  shape (L=4, D=64) 向量序列
       │
       │  + 位置编码 → N × Transformer block (self-attention + FFN)
       ▼
  shape (L=4, D=64) 已被「混合」的向量
       │
       │  取最后一个位置 → Linear(d_model, vocab)
       ▼
  shape (vocab,) logits
       │
       │  softmax / 采样
       ▼
  next token id → tokenizer.decode → '中文'
```

## 1. Tokenizer：字符串 → token ids

**三种主流粒度**：
- **字符级**：vocab 小（几千），序列长，模型难学语义
- **词级**：vocab 大（几十万），OOV（生词）问题严重
- **子词级 BPE / WordPiece / SentencePiece**：vocab 适中（3-15 万），把常见词当一个 token，罕见词拆字节 —— **现代 LLM 全用这个**

下面用 `tiktoken`（OpenAI 的 BPE 实现）看 GPT-4 用的 `cl100k_base` 分词器怎么处理中英混合。

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')      # GPT-4 用的 BPE tokenizer
print('vocab size:', enc.n_vocab)

samples = [
    'I love NLP',
    '我爱 NLP',
    '注意力机制 (Attention) 是 Transformer 的核心',
    'def softmax(x): return np.exp(x) / np.exp(x).sum()',
]
for s in samples:
    ids = enc.encode(s)
    pieces = [enc.decode([i]) for i in ids]
    print(f'\n输入: {s!r}')
    print(f'  token 数: {len(ids)}')
    print(f'  ids    : {ids}')
    print(f'  pieces : {pieces}')

In [ ]:
# 观察：中文比英文「token 更碎」—— 同样的语义中文要花更多 token
import statistics

en = 'Attention is all you need. This paper introduces the Transformer architecture.'
zh = '注意力就是你所需要的一切。这篇论文引入了 Transformer 架构。'

for label, txt in [('英文', en), ('中文', zh)]:
    ids = enc.encode(txt)
    print(f'{label}：字符 {len(txt):3d}，token {len(ids):3d}，平均每 token {len(txt)/len(ids):.2f} 字符')

print('\n→ 含义：中文场景用 GPT 系列 API，相同意思花的 token 通常多 2~3 倍。')
print('→ 这也是为什么国产模型用中文友好的 tokenizer（Qwen / DeepSeek 等）。')

## 2. Embedding：token id → 高维向量

**心智模型**：vocab × d_model 的大查找表。每个 token id 对应一行向量。

**关键点**：这张表是**可学习**的参数。训练前是随机数，训练后语义相近的词的向量在空间里相近。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
vocab_size = enc.n_vocab            # 100277
d_model = 64                        # 向量维度（真实 LLM 通常 768~4096+）
embed = nn.Embedding(vocab_size, d_model)
print('Embedding 表参数量:', sum(p.numel() for p in embed.parameters()), '(=', vocab_size, '×', d_model, ')')

# 编码一句话
ids = torch.tensor(enc.encode('I love NLP'))
print('\ntoken ids:', ids.tolist())

vecs = embed(ids)
print('embedding 输出 shape:', vecs.shape, '  <- (L, D) 每个 token 一个 D 维向量')
print('第一个 token 的前 8 维:', vecs[0, :8].detach().numpy())

In [ ]:
# 训练前：随机 embedding，语义近的词向量也不近
import torch.nn.functional as F

pairs = [('king', 'queen'), ('king', 'banana'), ('cat', 'dog')]
for w1, w2 in pairs:
    id1 = torch.tensor(enc.encode(w1))
    id2 = torch.tensor(enc.encode(w2))
    v1 = embed(id1).mean(0)        # 多 token 的词取平均（演示用，真实场景另说）
    v2 = embed(id2).mean(0)
    sim = F.cosine_similarity(v1, v2, dim=0)
    print(f'cos({w1:6}, {w2:7}) = {sim.item():+.4f}')

print('\n→ 这些数字接近 0，因为 embedding 还没训练，只是随机初值。')
print('→ 真正训出来后：cos(king, queen) 会显著高于 cos(king, banana)。')

## 3. 位置编码：告诉模型「谁在前谁在后」

**问题**：self-attention 是 `集合操作`，把 token 顺序换了输出不变。需要显式注入位置信息。

**主流做法**：
- 绝对位置（原始 Transformer 用 sin/cos）
- 可学习位置（GPT-2）
- 相对位置 / RoPE（Llama、Qwen 等现代 LLM）

In [ ]:
import math

def sinusoidal_pe(max_len: int, d_model: int):
    """原始 Transformer 的位置编码：偶数维 sin，奇数维 cos，频率指数衰减。"""
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()                  # (L, 1)
    div = torch.exp(torch.arange(0, d_model, 2).float()
                    * -(math.log(10000.0) / d_model))                 # (D/2,)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(50, 64)
print('位置编码 shape:', pe.shape)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3))
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel('embedding dim')
plt.ylabel('position')
plt.title('Sinusoidal Positional Encoding (50, 64)')
plt.show()

In [ ]:
# 把位置加到 embedding 上：模型同时看到「词义 + 位置」
ids = torch.tensor(enc.encode('I love NLP'))
vecs = embed(ids)                              # (L, D)
vecs_with_pos = vecs + pe[:len(ids)]            # 广播：直接相加

print('vecs           shape:', vecs.shape)
print('vecs + position shape:', vecs_with_pos.shape, '<- 形状不变，但每个向量已包含位置信息')

## 4. 看一眼 Transformer Block（self-attention + FFN）

中间 N 层 Transformer block 把每个 token 的向量「与其它 token 的向量加权混合」，再经过 FFN 非线性变换。

这里只演示**最小可跑骨架**，下一个 notebook（07）会专门把 self-attention 拆开手撸。

In [ ]:
# 用 PyTorch 自带的 TransformerEncoderLayer 一行搞定（学概念够用）
torch.manual_seed(0)
layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, batch_first=True)
encoder = nn.TransformerEncoder(layer, num_layers=2)

x = vecs_with_pos.unsqueeze(0)            # (B=1, L, D)
y = encoder(x)
print('输入 :', x.shape)
print('输出 :', y.shape, '<- shape 不变，每个位置的向量已被「上下文化」')

## 5. 输出头：向量 → vocab 上的概率分布

**最后一步**：拿最后一个位置的向量（或所有位置，取决于任务），过一个 `Linear(d_model → vocab_size)` 得到 logits，softmax 后是「下一个 token」的概率分布。

**采样**：从分布里选一个 id —— greedy / top-k / top-p / temperature 各种策略（见 notebook 08）。

In [ ]:
lm_head = nn.Linear(d_model, vocab_size, bias=False)

last_vec = y[0, -1]                        # 最后一个位置 (D,)
logits = lm_head(last_vec)                  # (vocab_size,)
probs = torch.softmax(logits, dim=-1)
print('logits shape:', logits.shape)
print('概率和:', probs.sum().item())

# greedy: 选概率最大的 id
next_id = int(probs.argmax())
print(f'\ngreedy next id: {next_id}, decode = {enc.decode([next_id])!r}')

# top-5
top_probs, top_ids = probs.topk(5)
print('\ntop-5 候选（注意：模型还没训练，输出是随机的）:')
for p, i in zip(top_probs.tolist(), top_ids.tolist()):
    print(f'  id={i:6d} ({enc.decode([i])!r:>15})  p={p:.6f}')

## 深入思考

1. **为什么用同一个 embedding 表做输入和输出？**
   - 实际很多模型这么做（叫 **tied embeddings** / weight tying），省一半参数。GPT-2、Qwen 都用了。
2. **为什么 attention 之前要先 + 位置编码，而不是 concat？**
   - 加性比拼接省参数；数学上加性等价于「让 embedding 空间里多了一个表示位置的子空间」。
3. **如果输入 1000 token，输出 logits 是 (1000, vocab) 还是 (vocab,)？**
   - 训练时是 (1000, vocab) —— 每个位置都预测下一个 token，叫 **teacher forcing**。
   - 推理时只关心最后一个位置：(vocab,)。
4. **`temperature=0` 等价于 greedy 吗？**
   - 工程上是。数学上 `softmax(x/T)` 当 `T→0` 时退化成 one-hot 在 argmax 位置。

改一改：把 `enc.encode('hello world hello world')` 跑一下，看相同词在不同位置的 token id 是不是一样（应该一样，因为 tokenizer 跟位置无关）。

## 自检 ✅

- [ ] 画出从 string 到 next token 的 7 步流程图，每步说出形状变化。
- [ ] 解释「BPE 子词分词比字符级 / 词级好在哪」。
- [ ] 解释「为什么需要位置编码」。
- [ ] 给一个 vocab=50k、d=512 的模型，估算 embedding 表参数量（应 ≈ 25.6 M）。
- [ ] 解释「中文用 GPT tokenizer 比英文费 token 的根本原因」。

## 下一步

进入 Stage 2 → [`../stage2_进阶/06_mnist_mlp.ipynb`](../stage2_进阶/06_mnist_mlp.ipynb)